In [1]:
import os
import sys

# 设置缓存目录路径
cache_dir = r'F:\Anaconda\envs\pytorch\cache'

# 确保目录存在
os.makedirs(cache_dir, exist_ok=True)

# 设置所有相关环境变量
os.environ['HF_HOME'] = cache_dir
os.environ['TRANSFORMERS_CACHE'] = cache_dir
os.environ['HUGGINGFACE_HUB_CACHE'] = cache_dir

print(f"设置的缓存目录: {cache_dir}")

# 现在导入transformers
from transformers import pipeline
from transformers.utils import TRANSFORMERS_CACHE

print(f"实际缓存路径: {TRANSFORMERS_CACHE}")

# 验证路径是否正确
if cache_dir == TRANSFORMERS_CACHE:
    print("✅ 缓存路径设置成功！")
else:
    print(f"❌ 缓存路径设置失败！期望: {cache_dir}, 实际: {TRANSFORMERS_CACHE}")

# 运行情感分析
classifier = pipeline("sentiment-analysis")
result = classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

print(f"\n分析结果: {result}")

# 检查缓存目录是否创建了文件
print(f"\n缓存目录内容:")
if os.path.exists(cache_dir):
    items = os.listdir(cache_dir)
    if items:
        for item in items:
            item_path = os.path.join(cache_dir, item)
            if os.path.isdir(item_path):
                print(f"  📁 {item}")
            else:
                print(f"  📄 {item}")
    else:
        print("   (空目录)")
else:
    print("   (目录不存在)")

import os

# 检查并修复缓存目录权限
cache_dir = r"F:\Anaconda\envs\pytorch\cache"
if os.path.exists(cache_dir):
    import subprocess
    # 给当前用户完全控制权限
    subprocess.run(['icacls', cache_dir, '/grant', 'Everyone:F', '/T'], shell=True)

设置的缓存目录: F:\Anaconda\envs\pytorch\cache


f:\Anaconda\envs\pytorch\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


实际缓存路径: F:\Anaconda\envs\pytorch\cache
✅ 缓存路径设置成功！



Device set to use cuda:0



分析结果: [{'label': 'POSITIVE', 'score': 0.9598046541213989}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

缓存目录内容:
  📁 .locks
  📁 datasets
  📁 datasets--glue
  📁 evaluate
  📁 metrics
  📁 models--bert-base-cased
  📁 models--bert-base-uncased
  📁 models--camembert-base
  📁 models--distilbert--distilbert-base-uncased-finetuned-sst-2-english
  📁 models--WenjieSHI--dummy-model
  📁 modules
  📁 xet


使用 FAISS 进行语义搜索

在 第5小节 ，我们创建了一个 🤗 Datasets 仓库的 GitHub issues 和评论组成的数据集。在本节，我们将使用这些信息构建一个搜索引擎，帮助我们找到关于该库的最相关的 issue 的答案！

使用文本嵌入进行语义搜索

正如我们在 第一章 ，学习的，基于 Transformer 的语言模型会将文本中的每个 token 转换为嵌入向量。事实证明，我们可以 “池化（pool）” 嵌入向量以创建整个句子、段落或（在某些情况下）文档的向量表示。然后，通过计算每个嵌入之间的点积相似度（或其他一些相似度度量）并返回相似度最大的文档，这些嵌入可用于在语料库中找到相似的文档。

在本节中，我们将使用文本嵌入向量来开发语义搜索引擎。与基于将查询中的关键字的传统方法相比，这些搜索引擎具有多种优势。

加载和准备数据集

首先，我们需要下载我们的 GitHub issues 数据集，所以让我们像往常那样使用 load_dataset() 函数：

In [2]:
from datasets import load_dataset

issues_dataset = load_dataset("lewtun/github-issues", split="train")

issues_dataset

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


datasets-issues-with-comments.jsonl:   0%|          | 0.00/12.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3019 [00:00<?, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'timeline_url', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 3019
})

在此，我们在 load_dataset() 中指定了默认的 train（训练集） 部分，因此它返回一个 Dataset 而不是 DatasetDict 。首要任务是排除掉拉取请求（pull），因为这些请求往往很少用于回答提出的 issue，会为我们的搜索引擎引入干扰。正如我们现在熟悉的那样，我们可以使用 Dataset.filter() 函数来排除数据集中的这些行。与此同时，让我们也筛选掉没有评论的行，因为这些行没有为用户提问提供回答：

In [3]:
issues_dataset = issues_dataset.filter(
    lambda x: (x["is_pull_request"] == False and len(x["comments"]) > 0)
)
issues_dataset

Filter:   0%|          | 0/3019 [00:00<?, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'timeline_url', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 808
})

我们可以看到，我们的数据集中有很多列，其中大部分在构建我们的搜索引擎都不会使用。从搜索的角度来看，信息量最大的列是 title ， body ，和 comments ，而 html_url 为我们提供了一个回到原 issue 的链接。让我们使用 Dataset.remove_columns() 删除其余的列：

In [4]:
columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns_to_keep).symmetric_difference(columns)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 808
})

为了创建我们的文本嵌入数据集，我们将用 issue 的标题和正文来扩充每条评论，因为这些字段通常包含有用的上下文信息。因为我们的 comments 列当前是每个 issue 的评论列表，我们需要“重新组合”列，使得每一行都是由一个 (html_url, title, body, comment) 元组组成。在 Pandas 中，我们可以使用 DataFrame.explode() 函数 完成这个操作 它为类似列表的列中的每个元素创建一个新行，同时复制所有其他列值。让我们首先切换到 Pandas 的 DataFrame 格式：

In [5]:
issues_dataset.set_format("pandas")
df = issues_dataset[:]

如果我们检查这个 DataFrame 的第一行，我们可以看到这个 issue 有四个相关评论：

In [6]:
df["comments"][0].tolist()

['Cool, I think we can do both :)',
 '@lhoestq now the 2 are implemented.\r\n\r\nPlease note that for the the second protection, finally I have chosen to protect the master branch only from **merge commits** (see update comment above), so no need to disable/re-enable the protection on each release (direct commits, different from merge commits, can be pushed to the remote master branch; and eventually reverted without messing up the repo history).']

我们希望使用 explode() 将这些评论中的每一条都展开成为一行。让我们看看是否可以做到：

In [9]:
comments_df = df.explode("comments", ignore_index=True)
comments_df.head(10)

,html_url,title,comments,body
0,https://github.com/huggingface/datasets/issues...,Protect master branch,"Cool, I think we can do both :)",After accidental merge commit (91c55355b634d0d...
1,https://github.com/huggingface/datasets/issues...,Protect master branch,@lhoestq now the 2 are implemented.\r\n\r\nPle...,After accidental merge commit (91c55355b634d0d...
2,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,Hi ! I guess the caching mechanism should have...,## Describe the bug\r\nAfter upgrading to data...
3,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,"If it's easy enough to implement, then yes ple...",## Describe the bug\r\nAfter upgrading to data...
4,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,Well it can cause issue with anyone that updat...,## Describe the bug\r\nAfter upgrading to data...
5,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,"I just merged a fix, let me know if you're sti...",## Describe the bug\r\nAfter upgrading to data...
6,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,Definitely works on several manual cases with ...,## Describe the bug\r\nAfter upgrading to data...
7,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,Fixed by #2947.,## Describe the bug\r\nAfter upgrading to data...
8,https://github.com/huggingface/datasets/issues...,OSCAR unshuffled_original_ko: NonMatchingSplit...,I tried `unshuffled_original_da` and it is als...,## Describe the bug\r\n\r\nCannot download OSC...
9,https://github.com/huggingface/datasets/issues...,load_dataset using default cache on Windows ca...,"Hi @daqieq, thanks for reporting.\r\n\r\nUnfor...",## Describe the bug\r\nStandard process to dow...


非常好，我们可以看到其他三列已经被复制了，并且 comments 列里存放着单独的评论！现在我们已经完成了 Pandas 要完成的部分功能，我们可以通过加载内存中的 DataFrame 快速切换回 Dataset ：

In [10]:
from datasets import Dataset

comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 2964
})

我们获取到了几千条的评论！

 试试看！ 看看你是否可以使用 Dataset.map() 展开 issues_dataset 的 comments 列，这有点棘手；你可能会发现 🤗 Datasets 文档的 “批处理映射(Batch mapping)” 对这个任务很有用。

# 展平comments列，每个评论单独一行
def explode_comments(example):
    """将每条评论拆分为单独的行"""
    comments = example.get("comments", [])
    exploded_rows = []
    
    # 如果评论为空，保留原始行
    if not comments:
        return example
    
    # 为每条评论创建新行
    for comment in comments:
        # 复制原始行的所有字段
        new_row = example.copy()
        # 更新comment相关字段
        new_row["comment_body"] = comment.get("body", "")
        new_row["comment_user"] = comment.get("user", {}).get("login", "")
        new_row["comment_created_at"] = comment.get("created_at", "")
        new_row["comment_id"] = comment.get("id", "")
        exploded_rows.append(new_row)
    
    return exploded_rows

# 使用map函数展平comments列
exploded_dataset = issues_with_comments_dataset.map(
    explode_comments,
    batched=True,
    remove_columns=["comments"],  # 删除原始的comments列
    batch_size=100
)

# 将嵌套列表展平
from datasets import Dataset
exploded_dataset = Dataset.from_dict(exploded_dataset[:])

既然我们每行有一个评论，让我们创建一个新的 comments_length 列来存放每条评论的字数：

In [11]:
comments_dataset = comments_dataset.map(
    lambda x: {"comment_length": len(x["comments"].split())}
)

Map:   0%|          | 0/2964 [00:00<?, ? examples/s]

我们可以使用这个新列来过滤掉简短的评论，其中通常包括“cc @lewtun”或“谢谢！”之类与我们的搜索引擎无关的内容。筛选的精确数字没有硬性规定，但大约大于 15 个单词似乎是一个不错的选择：

In [12]:
comments_dataset = comments_dataset.filter(lambda x: x["comment_length"] > 15)
comments_dataset

Filter:   0%|          | 0/2964 [00:00<?, ? examples/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length'],
    num_rows: 2175
})

稍微清理了我们的数据集后，让我们使用 issue 标题、描述和评论构建一个新的 text 列。像往常一样，我们可以编写一个简单的函数，并将其传递给 Dataset.map() 来完成这些操作

In [13]:
def concatenate_text(examples):
    return {
        "text": examples["title"]
        + " \n "
        + examples["body"]
        + " \n "
        + examples["comments"]
    }


comments_dataset = comments_dataset.map(concatenate_text)

Map:   0%|          | 0/2175 [00:00<?, ? examples/s]

我们终于准备好创建一些文本嵌入了！让我们继续。

创建文本嵌入

正如我们在 第二章 学过，我们可以通过使用 AutoModel 类来完成文本嵌入。首先需要做的就是选择一个合适的 checkpoint。幸运的是，有一个名为 sentence-transformers 的库专门用于创建文本嵌入。如库中的 文档 所述的，我们这次要实现的是非对称语义搜索（asymmetric semantic search），因为我们有一个简短的查询，我们希望在比如 issue 评论等更长的文档中找到其匹配的文本。通过查看 模型概述表 我们可以发现 multi-qa-mpnet-base-dot-v1 checkpoint 在语义搜索方面具有最佳性能，因此我们将使用它。我们还将使用这个 checkpoint 加载了对应的 tokenizer ：

In [14]:
from transformers import AutoTokenizer, AutoModel

model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

将模型和输入的文本放到 GPU 上会加速嵌入过程，所以让我们现在就这么做：

In [15]:
import torch

device = torch.device("cuda")
model.to(device)

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

根据我们之前的想法，我们希望将我们的 GitHub issue 中的每一条记录转化为一个单一的向量，所以我们需要以某种方式“池化（pool）”或平均每个词的嵌入向量。一种流行的方法是在我们模型的输出上进行 CLS 池化 ，我们只需要收集 [CLS] token 的的最后一个隐藏状态。以下函数实现了这个功能：

In [16]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

接下来，我们将创建一个辅助函数，它将对一组文档进行 tokenize，然后将张量放在 GPU 上，接着将它们喂给模型，最后对输出进行 CLS 池化：

In [17]:
def get_embeddings(text_list):
    encoded_input = tokenizer(
        text_list, padding=True, truncation=True, return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

我们可以将第一条数据喂给它并检查输出的形状来测试这个函数是否正常工作：

In [18]:
embedding = get_embeddings(comments_dataset["text"][0])
embedding.shape

torch.Size([1, 768])

太好了，我们已经将语料库中的第一个条目转换为了一个 768 维向量！我们可以用 Dataset.map() 将我们的 get_embeddings() 函数应用到我们语料库中的每一行，然后创建一个新的 embeddings 列：

In [19]:
embeddings_dataset = comments_dataset.map(
    lambda x: {"embeddings": get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

Map:   0%|          | 0/2175 [00:00<?, ? examples/s]

请注意，在上面的代码中我们已经将文本嵌入转换为 NumPy 数组——这是因为当我们尝试使用 FAISS 搜索它们时，🤗 Datasets 需要这种格式，让我们继续吧。

使用 FAISS 进行高效的相似性搜索

现在我们有了一个文本嵌入数据集，我们需要一些方法来搜索它们。为此，我们将使用🤗 Datasets 中一种特殊的数据结构，称为 FAISS 指数。 FAISS （Facebook AI Similarity Search 的缩写）是一个库，提供了用于快速搜索和聚类嵌入向量的高效算法。

FAISS 背后的基本思想是创建一个特殊的数据结构，称为 index（索引） 它可以找到哪些嵌入与输入嵌入相似。在 🤗 Datasets 中创建一个 FAISS index（索引）很简单——我们使用 Dataset.add_faiss_index() 函数并指定我们要索引的数据集的哪一列：

In [27]:
embeddings_dataset.add_faiss_index(column="embeddings")

ImportError: You must install Faiss to use FaissIndex. To do so you can run `conda install -c pytorch faiss-cpu` or `conda install -c pytorch faiss-gpu`. A community supported package is also available on pypi: `pip install faiss-cpu` or `pip install faiss-gpu`. Note that pip may not have the latest version of FAISS, and thus, some of the latest features and bug fixes may not be available.

现在，我们可以使用 Dataset.get_nearest_examples() 函数进行最近邻居查找。让我们通过首先嵌入一个 issue 来测试这一点，如下所示：

In [28]:
question = "How can I load a dataset offline?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

(1, 768)

就像对文档进行嵌入一样，我们现在有一个表示查询的 768 维向量，我们可以将其与整个语料库进行比较以找到最相似的嵌入：

In [ ]:
scores, samples = embeddings_dataset.get_nearest_examples(
    "embeddings", question_embedding, k=5
)

Dataset.get_nearest_examples() 函数返回一个元组，包括评分（评价查询和文档之间的相似程度）和对应的样本（这里是 5 个最佳匹配）。让我们把这些收集到一个 pandas.DataFrame ，这样我们就可以轻松地对它们进行排序：

In [ ]:
import pandas as pd

samples_df = pd.DataFrame.from_dict(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)

现在我们可以遍历前几行来查看我们的查询与评论的匹配程度如何：

In [ ]:
for _, row in samples_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.scores}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()

不错！我们的输出的第 2 个结果似乎与查询匹配。

试试看！创建你自己的查询并查看你是否可以在检索到的文档中找到答案。你可能需要在 Dataset.get_nearest_examples() 增加参数 k 以扩大搜索范围。